# Fusion BEV : diagnostic sur quatre frames

Les frames 000006, 000036, 000052 et 000060 appartiennent au train commun (15 centres Car). Les poids camera et LiDAR sont audites sans exception legacy, puis leurs encodeurs restent geles, y compris la BatchNorm. Seuls la fusion, le backbone commun et la head centre/offset apprennent. Inputs complets, sans depth holdout. Evaluation sur les memes frames : memorisation, pas generalisation ni AP 3D. Ne pas reutiliser ces poids pour le pilote disjoint. Run All ne lance aucun entrainement par defaut.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / 'src/rgb_lidar_fusion').is_dir() and (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the repository or notebooks directory.')
sys.path.insert(0, str(ROOT / 'src'))
import torch
from rgb_lidar_fusion.bev_training import BEVFusionOverfitConfig, read_bev_overfit_split, prepare_bev_overfit_frames
from rgb_lidar_fusion.bev_model import BEVFusionDiagnostic
from rgb_lidar_fusion.bev_checkpoint import initialize_bev_fusion_encoders, load_bev_fusion_checkpoint
from rgb_lidar_fusion.bev_analysis import analyze_bev_fusion_frame, plot_bev_fusion_sample
print(f'ROOT={ROOT}')

## Parametres
200 epochs, batch 1, soit 800 mises a jour. Le JSON garde un preset CPU de 100 epochs. Le notebook utilise CUDA uniquement dans le processus enfant, avec preflight a 2048 MiB ; ce seuil ne garantit pas que le calcul tienne en memoire. Aucun repli CPU ou retry automatique. EPOCHS est le total vise lors d'une reprise.

In [ ]:
RUN_OVERFIT = False
RESUME_EXISTING = False
DEVICE = 'cuda'
EPOCHS = 200
CONFIG_PATH = ROOT / 'configs/training/kitti_bev_fusion_overfit.json'
OUTPUT_DIR = ROOT / 'results/training/kitti_bev_fusion_overfit_4_cuda'
CHECKPOINT_PATH = OUTPUT_DIR / 'checkpoints/latest.pt'
LOG_PATH = OUTPUT_DIR.with_suffix('.jsonl')
ANALYSIS_DIR = ROOT / 'results/analysis/bev_fusion' / OUTPUT_DIR.name
values = json.loads(CONFIG_PATH.read_text())
for key in ('data_root', 'split_path', 'camera_checkpoint', 'lidar_checkpoint'):
    values[key] = str(ROOT / values[key])
values.update(output_dir=str(OUTPUT_DIR), device=DEVICE, epochs=EPOCHS)
config = BEVFusionOverfitConfig.from_mapping(values)
INSPECT_IDS = list(config.sample_ids[:2])
torch.set_num_threads(config.cpu_threads)
print(config.serialized())
print(f'run={RUN_OVERFIT} resume={RESUME_EXISTING} device={DEVICE} total_epochs={EPOCHS}')

## Audit et inspection CPU avant apprentissage
Verifie les IDs train, les empreintes, les configurations et les poids sources. Les encodeurs sont pre-entraines, mais la fusion, le contexte commun et la head sont initialises aleatoirement. RMS n'est pas un score objet. Les pixels camera contributeurs et les points LiDAR observes restent des supports distincts : le contexte peut avoir des activations en dehors de ces supports. Le runner repetera son propre audit avant tout calcul CUDA. Les figures sont aussi sauvegardees en PNG sous ANALYSIS_DIR ; chaque chemin est affiche.

In [ ]:
split = read_bev_overfit_split(config)
with torch.random.fork_rng(devices=[]), torch.device('cpu'):
    torch.random.default_generator.manual_seed(config.seed)
    preview_model = BEVFusionDiagnostic(config.model_config).eval()
pretraining = initialize_bev_fusion_encoders(
    preview_model, camera_checkpoint=config.camera_checkpoint,
    lidar_checkpoint=config.lidar_checkpoint, split=split)
print({name: {key: report.get(key) for key in ('epoch', 'sha256', 'provenance')}
       for name, report in pretraining.items() if isinstance(report, dict)})
frames = prepare_bev_overfit_frames(config, split)
print('train IDs:', [f.sample_id for f in frames], 'Car counts:', [len(f.centers_xy) for f in frames])
for frame in frames:
    if frame.sample_id in INSPECT_IDS:
        sample = analyze_bev_fusion_frame(preview_model, frame, score_threshold=config.score_threshold,
                                          top_k=config.top_k, match_radius_m=config.match_radius_m)
        print(frame.sample_id, sample.loss, sample.matching)
        fig = plot_bev_fusion_sample(sample, grid=config.grid)
        ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
        figure_path = ANALYSIS_DIR / f'initial_{frame.sample_id}.png'
        fig.savefig(figure_path, dpi=150, bbox_inches='tight')
        print(f'Saved figure: {figure_path}')
        display(fig)
        plt.close(fig)

## Entrainement explicite en sous-processus
Passer RUN_OVERFIT a True puis reexecuter les parametres et cette cellule. Logs par batch et par epoch, evaluations a 0, 1, puis toutes les 5 epochs et a la fin. latest.pt est sauvegarde aux evaluations : une interruption perd les mises a jour suivantes. RESUME_EXISTING=True reprend ce checkpoint sans recharger les poids sources. Le log JSONL est externe au dossier du run pour ne pas bloquer un demarrage neuf. Interrupt arrete le processus enfant ; sa sortie libere les ressources GPU.

In [ ]:
if RUN_OVERFIT:
    if not RESUME_EXISTING and OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
        raise ValueError('Output directory is not empty; choose a new run or resume explicitly.')
    command = [sys.executable, str(ROOT / 'scripts/train_bev_fusion_overfit.py'),
               '--config', str(CONFIG_PATH), '--device', DEVICE, '--epochs', str(EPOCHS),
               '--output-dir', str(OUTPUT_DIR)]
    if RESUME_EXISTING:
        if not CHECKPOINT_PATH.is_file():
            raise FileNotFoundError(CHECKPOINT_PATH)
        command += ['--resume-from', str(CHECKPOINT_PATH)]
    env = dict(os.environ, CUDA_LAUNCH_BLOCKING='1', PYTHONUNBUFFERED='1')
    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    with LOG_PATH.open('a' if RESUME_EXISTING else 'w') as log, subprocess.Popen(
        command, cwd=ROOT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1) as process:
        try:
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line.rstrip(), flush=True)
            if process.wait() != 0:
                raise RuntimeError(f'Training process failed with code {process.returncode}.')
        except BaseException:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
            raise
else:
    print('Training disabled. Set RUN_OVERFIT=True to start explicitly.')

## Courbes de memorisation
Les losses sont moyennes par frame. TP/FP/FN : maxima locaux, seuil 0.3, top-k 50, domaine annote et appariement glouton par score dans un rayon XY de 1 m. L'erreur metrique concerne les centres apparies seulement. Aucun centre apparie donne None, pas une erreur nulle. Une loss basse sans rappel correct ne suffit pas.

In [ ]:
metrics_path = OUTPUT_DIR / 'metrics.json'
metrics = json.loads(metrics_path.read_text()) if metrics_path.is_file() else []
if metrics:
    display(metrics[-1])
    epochs = [row['epoch'] for row in metrics]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for key in ('train_eval_loss', 'train_eval_heatmap', 'train_eval_offset'):
        axes[0].plot(epochs, [row[key] for row in metrics], label=key)
    axes[0].set(xlabel='Epoch', ylabel='Loss', yscale='log', title='Same training frames')
    for key in ('precision', 'recall'):
        axes[1].plot(epochs, [row[key] for row in metrics], label=key)
    axes[1].set(xlabel='Epoch', ylabel='Fraction', ylim=(-0.05, 1.05), title='Center matching, not 3D AP')
    for ax in axes:
        ax.legend()
        ax.grid(alpha=0.3)
    fig.tight_layout()
    display(fig)
    plt.close(fig)
else:
    print('No metrics yet. Training remains opt-in.')

## Inspection CPU du checkpoint sauvegarde
Cercles verts : centres annotes ; croix rouges : predictions dans le domaine annote. Comparer les supports, les RMS camera/LiDAR puis fusion/contexte commun et les scores avant/apres. Les echelles RMS sont automatiques par panneau et ne mesurent pas une confiance objet. Les pics memoire PyTorch et le temps du runner sont dans run_metadata.json, pas une mesure de latence d'inference ni de toute la VRAM.

In [ ]:
if CHECKPOINT_PATH.is_file():
    trained_model, payload = load_bev_fusion_checkpoint(CHECKPOINT_PATH)
    metadata = payload['training_metadata']
    if metadata['task'] != 'bev_fusion_overfit':
        raise ValueError('Expected the four-frame diagnostic, not a pilot checkpoint.')
    saved_config = BEVFusionOverfitConfig.from_mapping(metadata['runner_config'])
    saved_frames = prepare_bev_overfit_frames(saved_config, payload['split'])
    print(f"Checkpoint epoch={payload['epoch']} step={payload['step']}")
    print(metadata['evaluation_scope'], metadata['resources'])
    for frame in saved_frames:
        if frame.sample_id in INSPECT_IDS:
            sample = analyze_bev_fusion_frame(trained_model, frame, score_threshold=saved_config.score_threshold,
                                              top_k=saved_config.top_k, match_radius_m=saved_config.match_radius_m)
            print(frame.sample_id, sample.loss, sample.matching)
            fig = plot_bev_fusion_sample(sample, grid=saved_config.grid, weights_label=f"epoch {payload['epoch']}")
            ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
            figure_path = ANALYSIS_DIR / f"epoch_{payload['epoch']:04d}_{frame.sample_id}.png"
            fig.savefig(figure_path, dpi=150, bbox_inches='tight')
            print(f'Saved figure: {figure_path}')
            display(fig)
            plt.close(fig)
else:
    print('No trained checkpoint yet.')